In [ ]:
import os
import json
from image_processing import *
from config import *
import geopandas as gpd
from PIL import Image, ImageOps, ImageDraw,ImageFile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from mpl_toolkits.axes_grid1 import ImageGrid
from datetime import datetime, timedelta
import numpy.ma as ma
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
# import matplotlib.ticker as ticker
# from matplotlib.patches import FancyArrowPatch
# import matplotlib.patheffects as pe

In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
# os.path.basename(file_path[0]).split('_')[3]

In [ ]:
# phenocam_dir = '/home/ajai-krishna/work/Phenocam_d3/Phenocamdata_local'
data_dir=[os.path.join(phenocam_dir,files) for files in os.listdir(phenocam_dir) if files.endswith('.jpg')]
plot_dir= os.path.join(BASE_DIR+'Plots/')
file_path = [files for files in data_dir]
# file_path=[]
# for data in data_path:
#     if os.path.basename(data).split('_')[3] == '2026':
#         file_path.append(data)
# file_path

In [ ]:
# os.path.join(BASE_DIR+'Plots')

In [ ]:
df1 = create_image_metadata_df(file_path)
df1

In [ ]:
df_ndvi=df1[df1['image_type']=='ndvi'].reset_index(drop=True)
df_ndvi

In [ ]:
ndvi_img=np.array(Image.open(df_ndvi['file_path'][3]))
rows=cols=10
h, w = ndvi_img.shape[:2]
tile_h= h // rows
tile_w = w//rows
tiles = []
mean_ndvi = np.zeros((rows, cols))

for i in range(rows):
    for j in range(cols):
        tile = ndvi_img[
            i*tile_h:(i+1)*tile_h,
            j*tile_w:(j+1)*tile_w]
        tiles.append(tile)
        mean_ndvi[i,j]=np.nanmean(tile)

In [ ]:
ndvi_img.shape

In [ ]:
tiles[0].shape


In [ ]:
dates = df1['date'].unique()
selected_fils = []
for date in dates:
    group = df1[df1['date'] == date]
    ir_times = group[group['image_type'] == 'ir']['time'].values
    rgb_times = group[group['image_type'] == 'rgb']['time'].values
    common_times = set(ir_times).intersection(set(rgb_times))
    time1 = common_times.pop() if common_times else None
    if time1:
        ir_file = group[(group['image_type'] == 'ir') & (group['time'] == time1)]['file_path'].values[0]
        rgb_file = group[(group['image_type'] == 'rgb') & (group['time'] == time1)]['file_path'].values[0]
        selected_fils.append({'date': date, 
                              'time': time1, 
                              'ir_file':ir_file, 
                              'rgb_file':rgb_file})


In [ ]:
df=pd.DataFrame(selected_fils).sort_values(by=['date','time']).reset_index(drop=True)
df

In [ ]:

def select_time(group):
    ir_time = group[group['image_type'] == 'ir']['time'].values
    rgb_df = group[group['image_type'] == 'rgb']['time'].values
    for time1 in ir_time:
        if time1 in rgb_df:
            return time1    

df1['temp']= df1.groupby(['date']).apply(select_time)

In [ ]:
df1['temp'].unique()
df1

In [ ]:
df['ndvi'] = df.apply(
    lambda row: ndvi_calculation(row['ir_file'], row['rgb_file']),
    axis=1)


In [ ]:
save_ndvi_plot(df, plot_dir)